# Threads Bot — инструментарий для JSON-аккаунтов (Cloudflare)

Проверка, нормализация, импорт в Cloudflare D1.


## Ячейка 1 — загрузка JSON-файлов


In [ ]:
from google.colab import files
import json, time, os

uploaded = files.upload()
print(f"Загружено: {len(uploaded)} файлов")

## Ячейка 2 — проверка (валидность, ключевые cookies, срок)


In [ ]:
KEY = {"sessionid", "ds_user_id", "ig_did"}
for fname, data in uploaded.items():
    try:
        cl = json.loads(data.decode("utf-8"))
        assert isinstance(cl, list) and cl, "не массив"
        names = {c.get("name") for c in cl if isinstance(c, dict)}
        exps = [c.get("expirationDate") or c.get("expires") for c in cl if isinstance(c, dict) and (c.get("expirationDate") or c.get("expires") or 0) > 0]
        exp = min(exps) if exps else None
        ok = "OK" if names & KEY else "⚠️ нет sessionid"
        srok = time.strftime("%d.%m.%Y", time.localtime(exp)) if exp else "без срока"
        expired = "❌ ИСТЕКЛИ" if (exp and exp < time.time()) else "✅"
        print(f"{fname} | {ok} | {srok} | {expired}")
    except Exception as e:
        print(f"{fname} | ❌ {e}")

## Ячейка 3 — импорт в Cloudflare D1
Штатный способ (на локальной машине):
```bash
npm run accounts:import -- accounts --remote
```
Или сгенерируй SQL из загруженных файлов и выполни через wrangler.


In [ ]:
# Генерация SQL из загруженных JSON-файлов
sql_lines = []
now = time.strftime("%Y-%m-%dT%H:%M:%S.000Z", time.gmtime())
for fname in [f for f in os.listdir("/content") if f.endswith(".json")]:
    raw = open("/content/" + fname, encoding="utf-8").read()
    json.loads(raw)  # проверка
    name = os.path.splitext(fname)[0]
    q = lambda v: "'" + v.replace("'", "''") + "'"
    sql_lines.append(f"INSERT INTO threads_accounts(name,cookies,enabled,is_alive,hourly_reset,updated_at) VALUES({q(name)},{q(raw)},1,1,{q(now)},{q(now)}) ON CONFLICT(name) DO UPDATE SET cookies=excluded.cookies,enabled=1,is_alive=1,last_error=NULL,updated_at=excluded.updated_at;")
open("/content/accounts.sql", "w").write("\n".join(sql_lines))
print(f"SQL готов: {len(sql_lines)} аккаунтов")
print("\nВыполни на локальной машине:")
print("npx wrangler d1 execute threadsbot --remote --file=accounts.sql")

## Ячейка 4 — очистка


In [ ]:
for fname in list(os.listdir("/content")):
    if fname.endswith((".json", ".sql")):
        os.remove("/content/" + fname)
        print(f"Удалён: {fname}")
print("Готово.")

## Памятка по банам
1. Одна сессия = один IP одновременно.
2. Не поднимай лимит 20 запросов/час.
3. Cookies обновляются автоматически после каждого запроса.
4. Мёртвые сессии нужно пере-экспортировать вручную.
5. В боте: `/accounts`, `/account_check`, `/account_del`, `/account_export`.
